# Model-Derived Systemhood Benchmarks v0.13
All reported diagnostics are computed from declared finite-dimensional models. Missing physical definitions remain NaN/undefined; no hand-assigned scores are used.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
I=np.eye(2,dtype=complex); X=np.array([[0,1],[1,0]],complex); Z=np.array([[1,0],[0,-1]],complex)
zero=np.array([1,0],complex); one=np.array([0,1],complex); plus=(zero+one)/np.sqrt(2)
def dm(psi): return np.outer(psi,psi.conj())
def entropy(r):
    e=np.linalg.eigvalsh((r+r.conj().T)/2); e=e[e>1e-12]; return -np.sum(e*np.log2(e))
def ptrace2(r,keep):
    q=r.reshape(2,2,2,2)
    return np.trace(q,axis1=1,axis2=3) if keep==0 else np.trace(q,axis1=0,axis2=2)
def Cmi(r): return (entropy(ptrace2(r,0))+entropy(ptrace2(r,1))-entropy(r))/2
def autonomy(J,g,J0=.1): return abs(J)/(abs(J)+abs(g)+J0)
def binding(Esep,Estar=1.): return Esep/(Esep+Estar) if Esep is not None else np.nan
def fidelity_pure(r,psi): return float(np.real(np.vdot(psi,r@psi)))
bell=(np.kron(zero,zero)+np.kron(one,one))/np.sqrt(2)


## Operational definitions
B: normalized declared separation-energy proxy. C: normalized two-qubit mutual information. A: internal/external coupling autonomy J/(J+g+J0). P: one-step predictive fidelity under a declared effective model.

In [ ]:
rows=[]
def add(name,B,C,A,P): rows.append([name,B,C,A,P])
# 1 product free pair: no declared binding energy; no internal interaction; exact free effective prediction
rho=dm(np.kron(plus,plus)); add('product-free',np.nan,Cmi(rho),autonomy(0,0),1.0)
# 2 Bell pair after preparation, interaction switched off: correlation survives, no continuing internal coupling
rho=dm(bell); add('bell-off',np.nan,Cmi(rho),autonomy(0,0),1.0)
# 3 interacting isolated dimer; Esep is explicitly tied to |J| in this toy Hamiltonian proxy
for J in [.25,.6,1.0]:
    # choose an entangled family generated by XX evolution from |00> at fixed t*=0.55
    th=J*.55; psi=np.cos(th)*np.kron(zero,zero)-1j*np.sin(th)*np.kron(one,one); rho=dm(psi)
    add(f'dimer-J{J}',binding(abs(J)),Cmi(rho),autonomy(J,0),1.0)
# 4 same internal dimer with increasing environment coupling; P uses fidelity of an effective model that wrongly neglects a phase-flip event probability p
J=.6; th=J*.55; psi=np.cos(th)*np.kron(zero,zero)-1j*np.sin(th)*np.kron(one,one); r0=dm(psi); U=np.kron(Z,I)
for g,p in [(.15,.08),(.4,.2),(.8,.38)]:
    r=(1-p)*r0+p*(U@r0@U); add(f'open-g{g}',binding(abs(J)),Cmi(r),autonomy(J,g),fidelity_pure(r,psi))
for row in rows: print(row)


## Missingness is explicit

In [ ]:
names=['B','C','A','P']; Zphys=np.array([r[1:] for r in rows],float); labels=[r[0] for r in rows]
print('defined counts:',dict(zip(names,np.sum(np.isfinite(Zphys),axis=0))))
complete=np.all(np.isfinite(Zphys),axis=1); print('complete rows:',[labels[i] for i in np.where(complete)[0]])


## Geometry on complete rows only

In [ ]:
Zc=Zphys[complete]; labc=np.array(labels)[complete]; centered=Zc-Zc.mean(0); sv=np.linalg.svd(centered,compute_uv=False); print('complete matrix rank=',np.linalg.matrix_rank(Zc)); print('centered rank=',np.linalg.matrix_rank(centered)); print('singular values=',sv); print('condition=',sv[0]/sv[-1] if sv[-1]>1e-12 else np.inf)
print('correlation\n',np.round(np.corrcoef(Zc,rowvar=False),3))


## Pairwise identifiable subspaces
Rows with undefined B still contain valid information about C,A,P. Analyze available columns without imputing B.

In [ ]:
for cols in [('C','A'),('C','P'),('A','P'),('C','A','P')]:
    idx=[names.index(c) for c in cols]; Q=Zphys[:,idx]; ok=np.all(np.isfinite(Q),axis=1); Q=Q[ok]-Q[ok].mean(0); sv=np.linalg.svd(Q,compute_uv=False); print(cols,'rows=',len(Q),'rank=',np.linalg.matrix_rank(Q),'sv=',np.round(sv,4))


## Definition sensitivity
Autonomy contains reference scale J0. Scan it to expose how much feature geometry depends on this convention.

In [ ]:
for J0 in [.03,.1,.3,1.0]:
    vals=[autonomy(.6,g,J0) for g in [0,.15,.4,.8]]; print('J0',J0,' A=',np.round(vals,3))


## Interpretation
Unlike v0.12, these rows are generated from explicit toy definitions. The complete-row geometry is limited because the benchmark family is still structurally narrow: B is tied to J, A also depends on J, and open-system P is tied to the chosen dephasing model. Undefined B for free/Bell cases is retained rather than invented. The correct conclusion is not that the four-dimensional basis has been validated, but that a physically computable benchmark pipeline now exists and immediately exposes definition dependence and insufficient feature diversity.